# DEKRA source adapter

Sections 1–5 run the steps in `dekra.adapter` (`1_scrapers/dekra/src/adapter.py`) offline against the universe saved by the recon (`1_scrapers/dekra/test_output/2026-09-30_1831_recon/`, request R09: no type filter, world box). Section 6 fetches the finder (live) or simulates it. All logic lives in the module; this notebook only calls it. Every path comes from `core.paths`. DEKRA's API needs no key and no cookies.

**Scope rule (decided 2026-10-05).** A record is in scope if its country is DE, it is not a pure *WHG Standort*, and it is either typed *Automobil Prüfwesen und Gutachten* or untyped / AT-typed (*Automobil Prüfwesen*) with a vehicle-site word in its name (Station, Standort, Niederlassung, Außenstelle, Prüfstelle, Gutachten, GA-Büro, Automobil). **Nothing else is checked or excluded:** doubtful records are only flagged in `doubt_flags` and `needs_review`; validation happens later in the competitor map.

| Flag | Meaning |
|---|---|
| `untyped` / `at_type` | no Automobil type; in scope only because of the name |
| `thin` | no profile page, no booking link, no opening hours, no free text |
| `psp_suspect` | parenthesis or dealer / premises words in the name (possible workshop site) |
| `no_vehicle_disclaimer` | the text says it is not a Kfz-Prüfstelle / does no vehicle testing |
| `non_dekra_operator` | operator line is not DEKRA (partner office); informational, does not set `needs_review` |

## 1. Setup

In [ ]:
%load_ext autoreload
%autoreload 2

import pandas as pd

from core import paths
from core.geo import ensure_geonames, load_plz_centroids
from dekra import adapter as dekra

ROOT = paths.project_root()
RECON_RUN = "2026-09-30_1831_recon"  # run folder under 1_scrapers/dekra/test_output/
SAVED_RESPONSE = paths.scraper_dir("dekra") / "test_output" / RECON_RUN / "requests" / "r09_response.json"

# What a run writes. Minimal by default, to keep the folder small:
KEEP = ("pruefstellen_csv",)  # add "all_csv", "all_xlsx" or "pruefstellen_xlsx" if wanted
SAVE_RAW = False  # raw.jsonl is ~30 MB per full run; without it a run cannot be re-parsed offline later

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
ROOT

## 2. Load the saved universe

In [ ]:
from datetime import datetime

data = dekra.load_response(SAVED_RESPONSE)

# The fetch time is not stored in the file; its modification time is the best proxy.
scraped_at = datetime.fromtimestamp(SAVED_RESPONSE.stat().st_mtime).isoformat(timespec="seconds")

print("Records:", len(data["locations"]))
print("isNextLocation:", data["isNextLocation"])
print("scraped_at:", scraped_at)

## 3. Parse

In [ ]:
df = dekra.parse_records(data["locations"], scraped_at=scraped_at)
print(df.shape)
df[["location_id", "name", "zip", "city", "country", "dekra_types", "in_scope", "scope_reason", "doubt_flags"]].head()

### 3b. Enrich (GeoNames PLZ centroids)

Fills `state` from the PLZ, adds `plz_distance_km`, and flags German rows that are far from their PLZ or have an unknown PLZ. Downloads `DE.zip` into `4_reference/geonames/` on first use.

In [ ]:
centroids = load_plz_centroids(ensure_geonames(paths.geonames_dir()))
print(f"{len(centroids)} PLZ centroids")

df = dekra.enrich_german(df, centroids)  # PLZ checks for German rows only
df.loc[df["in_scope"], ["source_id", "zip", "city", "state", "plz_distance_km", "needs_review", "needs_review_reason"]].head()

## 4. Inspect

In [ ]:
in_scope = df[df["in_scope"]]
print(f"{len(in_scope)} of {len(df)} records in scope")
print()
print("Site types (in scope):", dict(in_scope["site_type"].value_counts()))
print()
print("Out of scope, by reason:")
df.loc[~df["in_scope"], "scope_reason"].map(lambda v: v.split(" (")[0]).value_counts()

In [ ]:
# Doubt flags among the in-scope rows (a row can carry several)
flags = in_scope["doubt_flags"].dropna().str.split("; ").explode().value_counts()
flags

In [ ]:
# In-scope rows per Bundesland
in_scope["state"].value_counts(dropna=False)

In [ ]:
review = in_scope[in_scope["needs_review"]]
print(f"{len(review)} in-scope rows are flagged for review (they stay in the export)")
review[["source_id", "name", "city", "doubt_flags", "site_type_evidence"]]

## 5. Export

Writes the saved universe's rows to a `recon` run folder of its own in `1_scrapers/dekra/test_output/` (`YYYY-MM-DD_HHMM_recon`). By default only `dekra_pruefstellen_<date>.csv` (the in-scope rows) is written, plus the small `run_summary.json`/`.md`; `KEEP` in section 1 chooses more (`all_csv` = every record DEKRA lists in all countries, and the XLSX copies). A rerun in the same minute gets the next free suffix (`_2`, `_3`, ...); nothing is ever overwritten. The files contain contact data and stay out of git.

In [ ]:
run_folder, written = dekra.write_recon_run(df, SAVED_RESPONSE, exports=KEEP)
for key, (path, rows) in written.items():
    print(f"{key:<19}{rows:>5} rows  {path.relative_to(ROOT)}")

## 6. Fetch

Every run writes one folder `YYYY-MM-DD_HHMM_<mode>` with the small `query_log.csv` and `run_summary.json`/`.md`, the exports chosen in `KEEP` (default: only `dekra_pruefstellen_<date>.csv`) and, if `SAVE_RAW = True`, `raw.jsonl`: full runs in `1_scrapers/dekra/output/`, sample and simulated runs in `1_scrapers/dekra/test_output/`.

**All six requests and all checks always run.** `KEEP` and `SAVE_RAW` (section 1; repeated in the self-contained cells) only decide what is written to disk. The checks use the full responses in memory. Without `raw.jsonl` a run cannot be re-parsed offline later; set `SAVE_RAW = True` for a run you want to keep as the source of truth

**What a full run requests (6 requests, ~30 MB, about a minute with the 5 s pauses).** All with `types: []` (no type filter, so untyped records come back too):

1. `world`: the whole map. This is the data source: some DE-coded records have coordinates outside Germany's box.
2. `de_box`: Germany's bounding box (lat 46.9–55.4, lng 5.5–15.4).
3. `q1`–`q4`: the four quarters of that box.

**Run status.** **PASS** needs all of: no error or abort, `isNextLocation` never `true`, no duplicate ids, and by record id: the quarters together equal the Germany box, the Germany box lies inside the world response, and the world records inside the box equal the Germany box. Anything else is **FAIL** and the export files carry `unverified` in their names. The drop check compares the in-scope count with the newest earlier full run in `output/` that passed (alert below −10 %; the first run reports "no baseline"). A sample (one quarter) is never a PASS.

**Politeness.** Sequential, 5 s + jitter between requests, 60 s timeout, one retry on timeout or connection error; the run stops on any 4xx/5xx, non-JSON answer or unexpected shape.

### 6a. Simulated run (no network)

100 synthetic records, the same six requests against a fake API. Must PASS.

In [ ]:
sim = dekra.run(mode="simulated", centroids=centroids, exports=KEEP)

### 6b. Sample run (real API, 1 request)

**Live request.** Self-contained: runs right after a kernel restart without the cells above. Fetches only quarter `q4` (south-east, ~1 MB), and exports it (raw response only if `SAVE_RAW = True`). No key needed. A sample always reports `SAMPLE`: it has no completeness proof.

In [ ]:
# Self-contained: runs right after a kernel restart, without the cells above
from core import paths
from core.geo import ensure_geonames, load_plz_centroids
from dekra import adapter as dekra

ROOT = paths.project_root()
# What a run writes. Minimal by default, to keep the folder small:
KEEP = ("pruefstellen_csv",)  # add "all_csv", "all_xlsx" or "pruefstellen_xlsx" if wanted
SAVE_RAW = False  # raw.jsonl is ~30 MB per full run; without it a run cannot be re-parsed offline later

centroids = load_plz_centroids(ensure_geonames(paths.geonames_dir()))

sample = dekra.run(mode="sample", root=ROOT, centroids=centroids, exports=KEEP, save_raw=SAVE_RAW)  # 1 live request
sample.frame.loc[sample.frame["in_scope"], ["name", "city", "state", "site_type", "doubt_flags"]].head(10)

### 6c. Full run

**Live requests (6).** DEKRA's finder needs no key, and automated collection is covered by the management approval of 2026-09-28. The full run needs `confirm_full_run=True`. The committed default is `False`: with it, the cell stops with an error on purpose before any request is sent.

Self-contained like 6b. Expect about 1,070 records for Germany's box and about 3,000 for the world (of which about 830 are in scope), but the figures come from the run: compare them with the summary, not with this text.

In [ ]:
# Self-contained: runs right after a kernel restart, without the cells above
from core import paths
from core.geo import ensure_geonames, load_plz_centroids
from dekra import adapter as dekra

ROOT = paths.project_root()
# What a run writes. Minimal by default, to keep the folder small:
KEEP = ("pruefstellen_csv",)  # add "all_csv", "all_xlsx" or "pruefstellen_xlsx" if wanted
SAVE_RAW = False  # raw.jsonl is ~30 MB per full run; without it a run cannot be re-parsed offline later

centroids = load_plz_centroids(ensure_geonames(paths.geonames_dir()))

# Set confirm_full_run=True to start the full run. Keep False in the committed notebook.
full = dekra.run(
    mode="full", confirm_full_run=False, root=ROOT, centroids=centroids, exports=KEEP, save_raw=SAVE_RAW
)

In [ ]:
# After a full run: completeness checks and the in-scope rows
print(full.summary["outcome"])
full.summary["completeness"]["checks"]